from the ‘traces’ dataset, select to_address, from_address, transaction_hash, and call_type values for every trace_id that appears over the observation window; download table locally as 'filtered_traces'.

In [ ]:
%%bigquery
SELECT
    trace_id,
    from_address,
    to_address,
    trace_type,
    call_type,
    `input`,
    block_number,

FROM `bigquery-public-data.crypto_ethereum.traces`
WHERE trace_type IN ('call', 'create')
    --AND to_address IS NOT NULL
    AND `bigquery-public-data.crypto_ethereum.traces`.block_timestamp > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 100 DAY)

from the ‘contracts’ dataset: extract address, bytecode, and funtion_sighashes columns; download table locally as 'filtered_contracts'.

In [ ]:
%%bigquery
SELECT
    address,
    bytecode,
    function_sighashes,

 FROM `bigquery-public-data.crypto_ethereum.contracts`
 WHERE `bigquery-public-data.crypto_ethereum.contracts`.bytecode <> '0x'
 GROUP BY address, bytecode, function_sighashes

limit from_address and to_address entries in filtered_traces based on their presence in filtered_contracts dataset; note their bytecode and function selectors, and save as 'contract_corpus'.

In [ ]:
%%sql
SELECT
    DISTINCT from_address.filtered_traces,
    bytecode.filtered_contracts,
    function_sighashes.filtered_contracts,

    DISTINCT to_address.filtered_traces,
    bytecode.filtered_contracts,
    function_sighashes.filtered_contracts,

FROM `filtered_traces`
JOIN `filtered_contracts`
    ON from_address = address OR to_address = address

proxy resolution using call_type field of filtered_traces

In [ ]:
%%sql
SELECT
    trace_id,
    from_address AS potential_proxy,
    to_address AS potential_implementation,
    trace_type,
    call_type,

FROM `filtered_traces`
WHERE call_type IN ('delegatecall', 'callcode')

--check if any of them were upgraded within the window by using this cell as a subquery for the cell below (more like joining filtered_traces on trace_type)

check bytecode sanity by checking if any contracts did a CREATE2 redeploy within the observation window

In [ ]:
%%sql
SELECT COUNT (trace_id)
FROM `filtered_traces`
WHERE trace_type = 'create'

--redeploys within the window may lead to misattribution of prevalence/usage metrics

total activity counts for each to_address in the filtered_traces table; denominators for dowmstream pattern prevalence/usage analysis

In [ ]:
%%sql
SELECT
  DISTINCT to_address as address,
  COUNT (DISTINCT transaction_hash) AS unique_txn_count, --the txn hash wasn't imported due to costs but can be obtained from the trace_id value (so this expression should carry a relation for obtaining txn_hash from trace_id
  COUNT (DISTINCT from_address) AS unique_caller_count,
  COUNT (trace_id) AS call_count

FROM `filtered_traces`
WHERE to_address IS NOT NULL --maybe change to the burn address value
GROUP BY to_address
ORDER BY unique_caller_count DESC, unique_txn_count DESC, call_count DESC;

rolled-up activity counts per unique bytecode. the bytecode column of this cell should be fed into the chosen disassembler for phase 2